<h1 align="center">Práctica 4. Árboles binarios de búsqueda</h1>
<h2 align="center">Sesión 2: Construcción equilibrada y autocompletado</h2>
<h3 align="center">Estructuras de datos</h3>
<h3 align="center">Grado en Inteligencia Artificial</h3>
<h3 align="center">2026-2027 · Universitat Politècnica de València</h3>


**Pon/poned aquí tú/vuestros nombre(s):**
- (sustituir por el nombre)
- (sustituir por el nombre o borrar si se realiza individualmente)

## Índice
1. ### [Actividad 1: Recorrer un `BST` de mayor a menor](#act1)
1. ### [Actividad 2: Cargar un diccionario como árbol balanceado](#act2)
1. ### [Actividad 3: Simular la función de autocompletado](#act3)


<a id='act1'></a>
## Actividad 1: Recorrer un `BST` de mayor a menor

A continuación hemos dejado el código de la clase `BST` adaptado a la representación de un *conjunto*. Para ello hemos eliminado el campo `_value` de la clase `BST._Node` y hemos sustituido el método `__getitem__` por `add`:

In [52]:
from operator import indexOf


class BST:

    class _Node:
        def __init__(self, key, left=None, right=None):
            self._key = key
            self._left = left
            self._right = right

    def __init__(self): # create empty binary tree
        self._root = None
        self._size = 0

    def __len__(self):
        return self._size

    def __contains__(self, key):
        node = self._root
        while node is not None:
            if key == node._key:
                return True
            elif key < node._key:
                node = node._left
            else:
                node = node._right
        return False

    def add(self, key):
        prev, node = None, self._root
        while node is not None:
            prev = node
            if key == node._key:
                return
            elif key < node._key:
                node = node._left
            else:
                node = node._right
        newchild = BST._Node(key)
        self._size += 1
        if prev is None:
            self._root = newchild
        elif key < prev._key:
            prev._left = newchild
        else:
            prev._right = newchild

    def height(self):

        def aux(node):
            if (node is None or
                (node._left is None and node._right is None)):
                return 0
            return 1 + max(aux(node._left), aux(node._right))

        return aux(self._root)

    def find_gt_iter(self, key): # sucesor iterativo
        def aux(nodo):
            """
            es como un inorder pero únicamente visita claves > key
            """
            if nodo is not None:
                if nodo._key > key:
                    yield from aux(nodo._left)
                    yield nodo._key
                yield from aux(nodo._right)
        yield from aux(self._root)

    def _inorder_traversal(self, node):
        if node is not None:
            yield from self._inorder_traversal(node._left)
            yield node
            yield from self._inorder_traversal(node._right)

    def __iter__(self):
        for node in self._inorder_traversal(self._root):
            yield node._key

    def __repr__(self):
        aux = []
        for node in self._inorder_traversal(self._root):
            aux.append(repr(node._key))
        return "{" + ",".join(aux) + "}"

    def lista_al_reves(self):
        resultado = []
        def aux(nodo):
            if nodo is None:
                return
            aux(nodo._right)
            resultado.append(nodo._key)
            aux(nodo._left)
        aux(self._root)
        #
        # COMPLETAR, IMPRESCINDIBLE RECORRER EL ÁRBOL DE MAYOR A MENOR
        # NO SIRVE RECORRELO DE MENOR A MAYOR Y LUEGO HACER REVERSE!!!
        #
        return resultado

Se pide completar el método `lista_al_reves` de manera que recorra todo el árbol **de mayor a menor**. Si, por ejemplo, ejecutamos el siguiente código:

```python
b = BST()
for i in (30, 10, 20, 5, 15, 25, 40):
    b.add(i)

b.lista_al_reves()
```

El resultado correcto será:

```python
[40, 30, 25, 20, 15, 10, 5]
```

**Nota:** El método se limita a rellenar una lista python. Puedes utilizar `yield` pero no se exige hacerlo así (es posible utilizar directamente un `append` donde corresponda).

In [12]:
b = BST()
for i in (30, 10, 20, 5, 15, 25, 40):
    b.add(i)

b.lista_al_reves()

[40, 30, 25, 20, 15, 10, 5]

<a id='act2'></a>
## Actividad 2: Cargar un diccionario como árbol balanceado

En esta segunda actividad se pide cargar todas las palabras del fichero `10000.txt` (las 10000 palabras más frecuentes del castellano, según la [RAE](http://corpus.rae.es/lfrecuencias.html)) y meterlas en un árbol binario de búsqueda.

In [13]:
palabras = []
with open('10000.txt') as f:
    palabras = f.read().split()
print(f"Hemos leído {len(palabras)} palabras")

Hemos leído 10000 palabras


Vamos a ver el tamaño y la altura de un BST tras cargar esas palabras:

In [22]:
%%time
# la línea de arriba es propio de jupyter para medir el tiempo de ejecución de esta celda

dicci = BST()

for pal in palabras:
    dicci.add(pal)

# calcular y mostrar el tamaño y la altura del árbol generado:
print(f"Tiene {len(dicci)} nodos y una altura igual a {dicci.height()}")

Tiene 10000 nodos y una altura igual a 1521
CPU times: user 427 ms, sys: 3.01 ms, total: 430 ms
Wall time: 430 ms


Altura 1521 significa que hay búsquedas que van a realizar ese número de pasos comparando y bajando ¿se puede hacer mejor o peor?

Vamos a probar 3 opciones:

1. Insertar los valores ordenados es de lo peorcito que podemos hacer.
2. ¿Y si insertamos los valores ordenadamente?
3. Vas a comparar con una inserción para conseguir el árbol más equilibrado posible

### Inserción de valores de menor a mayor

In [24]:
%%time
# la línea de arriba es propio de jupyter para medir el tiempo de ejecución de esta celda

# la lista ordenada:
ordenadas = sorted(palabras)

# sin estas 2 líneas daría RecursionError: maximum recursion depth exceeded
import sys
sys.setrecursionlimit(10_100)

dicci = BST()
# COMPLETAR PARA INSERTAR EN ORDEN LAS PALABRAS DE "ordenadas"
for pal in ordenadas:
    dicci.add(pal)
# calcular y mostrar el tamaño y la altura del árbol generado:
print(f"Tiene {len(dicci)} nodos y una altura igual a {dicci.height()}")

Tiene 10000 nodos y una altura igual a 9999
CPU times: user 2.31 s, sys: 25.1 ms, total: 2.34 s
Wall time: 2.39 s


### Inserción de los valores aleatoriamente

In [25]:
%%time
# la línea de arriba es propio de jupyter para medir el tiempo de ejecución de esta celda

import random
desordenadas = palabras.copy() # primero hacemos una copia independiente
random.shuffle(desordenadas) # luego se desordena

dicci = BST()
# COMPLETAR PARA INSERTAR EN ORDEN LAS PALABRAS DE "desordenadas"
for pal in desordenadas:
    dicci.add(pal)

# calcular y mostrar el tamaño y la altura del árbol generado:
print(f"Tiene {len(dicci)} nodos y una altura igual a {dicci.height()}")

Tiene 10000 nodos y una altura igual a 28
CPU times: user 16.6 ms, sys: 1.5 ms, total: 18.1 ms
Wall time: 17.3 ms


Si ejecutas la celda de arriba varias veces verás que no siempre da la misma altura. En las pruebas que hemos realizado, la altura suele salir entre 27 u 32.

### Ejercicio:

Compara el tiempo de ejecución de esta celda con el de las dos celdas anteriores:

- ¿Cuántas veces es más lento insertando los datos de forma ordenada?
- ¿Crees que ese ratio sería el mismo si en lugar de insertar 10000 palabras insertaras 5000? (puedes probarlo y sales de dudas)

### Inserción para conseguir un árbol lo más equilibrado posible

Para que quede balanceado, se propone crear una clase `BSTce` (`ce` de `c`onstrucción `e`quilibrada) que hereda de `BST` y que redefine el constructor `__init__` para que reciba una lista de valores que luego debe insertar en el árbol de manera que éste quede lo más equilibrado posible.

Para ello, tras ordenar la lista de valores recibida (no asumas que llega ordenada), va insertando en primer lugar el elemento de la posición central (puede utilizar el método `add`) y, de manera recursiva, aplica esa misma idea a cada mitad recursivamente.

Si la construcción equilibrada funciona correctamente, la altura del diccionario resultante será de 13.

In [50]:
class BSTce(BST): # observa que hereda de BST, podemos acceder a sus métodos y atributos
    def __init__(self, datosIniciales=()):
        super().__init__() # llamamos al constructor de BST
        listaOrdenada = sorted(x for x in datosIniciales)
        # llamada inicial al método auxiliar de abajo:
        self._inserta_lista_ordenada(listaOrdenada, 0, len(listaOrdenada)-1)
        
    def _inserta_lista_ordenada(self, thelist, first, last):
        if first > last:
            return
        mid = (first + last) // 2
        self.add(thelist[mid])
        self._inserta_lista_ordenada(thelist, first, mid - 1)
        self._inserta_lista_ordenada(thelist, mid + 1, last)

Vamos a probar la nueva clase a ver qué altura sale con los mismos datos:

In [51]:
%%time
# la línea de arriba es propio de jupyter para medir el tiempo de ejecución de esta celda)

dicci = BSTce(palabras)

# calcular y mostrar el tamaño y la altura del árbol generado:
print(f"Tiene {len(dicci)} nodos y una altura igual a {dicci.height()}")

Tiene 10000 nodos y una altura igual a 13
CPU times: user 13.7 ms, sys: 4.31 ms, total: 18 ms
Wall time: 17.1 ms


### Ejercicio:

Compara el tiempo de ejecución y la altura del árbol resultante de esta celda con el de las anteriores. ¿A cuál de las versiones anteriores se parece más?

<a id='act3'></a>
## Actividad 3: Simular la función de autocompletado

En esta actividad debes simular la función de autocompletado que tienen algunas aplicaciones como editores de texto, etc.

Para ello debes añadir:

- Un método llamado `find_ge_iter` que sea similar a `find_gt_iter` donde el `gt` de *greater than* se ha sustituido por `ge` de *greater or equal*. Es decir, recorre de menor a mayor los valores del árbol que sean mayoro o igual al argumento recibido.

- Un método llamado `autocompletado` en la clase `BSTautocompletado` (que hereda de `BSTce`) que recibe un prefijo `pref` y un número `n` (con valor por defecto igual a `10`) y que devuelve **una lista Python** con hasta `n` palabras (menos si no hay tantas) de manera que sean mayor o igual a `pref` y que siguan teniendo a `pref` **como prefijo.**

Por ejemplo, si el usuario introduce palabra `'casa'`, el método `autocompletado` debe devolver la lista de las hasta 10 primeras palabras (en orden alfabético) que puedan completar `'casa'` para formar una palabra:

```python
['casa', 'casada', 'casado', 'casarse', 'casas']
```

Otro ejemplo: el resultado de llamar a `autocompletado` con la palabra `'circ'` será:

```python
['circo',
 'circuito',
 'circuitos',
 'circulación',
 'circular',
 'circunstancia',
 'circunstancias']
```

pero si quieres autocompletar `'circo'` sólo será ella misma (el conjunto de palabras no contiene otras que empiecen por el prefijo `circo`):


```python
['circo']
```

Finalmente, si la palabra utilizada no permite completar ninguna palabra (ejemplo: `'xy'`), el resultado será la lista vacía:

```python
[]
```

### Sugerencia:

Recordad que las cadenas Python disponen el método `startswith` para determinar si una cadena es prefijo de otra. Por ejemplo, la siguiente línea:

```python
print('camarero'.startswith('cama'))
```
imprime `True`

mientras que:

```python
print('camarero'.startswith('hola'))
```
imprime `False`


In [59]:
class BSTautoc(BSTce):
    
    def find_ge_iter(self, key):
        #
        # COMPLETAR
        def aux(nodo):
            if nodo is not None:
                if nodo._key >= key:
                    yield from aux(nodo._left)
                    yield nodo._key
                yield from aux(nodo._right)
        yield from aux(self._root)
        #
        
    def autocompletado(self, prefijo, n=10):
        resultado = []
        #
        # COMPLETAR
        for word in self.find_ge_iter(prefijo):
            if word[:len(word)] == prefijo:
                resultado.append(word)
            if len(resultado) == n:
                return resultado
        return resultado
        
with open('10000.txt') as f:
    palabras = f.read().split()
               
dicci = BSTautoc(palabras)

El siguiente código sirve para comprobar que tu implementación del método `autocompletado` es correcto.

Por ejemplo, si introduces palabra `'casa'` devolverá:

```
Opciones de autocompletado para 'casa': casa, casada, casado, casarse, casas
```

Otro ejemplo:

```
Opciones de autocompletado para 'cama': cama, camacho, camarero, camas
```

Mientras que si introduces `'jarl'` el resultado será:

```python
no sé formar ninguna palabra con 'jarl'
```

In [60]:
prefijo = input()
opciones = dicci.autocompletado(prefijo)
if len(opciones)==0:
    print(f"no sé formar ninguna palabra con '{prefijo}'")
else:
    print(f"Opciones de autocompletado para '{prefijo}': {', '.join(opciones)}")

Opciones de autocompletado para 'casa': casa, casada, casado, casarse, casas


El siguiente código realiza una pequeña comprobación con algunas palabras de prueba:

In [61]:
ok = True
for prefijo,n,output in [('casa',5,['casa', 'casada', 'casado', 'casarse', 'casas']),
                         ('cama',3,['cama','camacho','camarero']),
                        ('circo',100,['circo']),
                        ('xyz',12,[])]:
    opciones = dicci.autocompletado(prefijo,n)
    if opciones == output:
        print("OK",prefijo,n,output)
    else:
        print('FAILED FOR',prefijo,n,"->",opciones,"instead of",output)
        ok = False

if ok:
    print("Tests pasados satisfactoriamente")

OK casa 5 ['casa', 'casada', 'casado', 'casarse', 'casas']
OK cama 3 ['cama', 'camacho', 'camarero']
OK circo 100 ['circo']
OK xyz 12 []
Tests pasados satisfactoriamente
